In [20]:
import pandas as pd

# ==============================
# LOAD DATA
# ==============================
jan_df = pd.read_csv("jan_aushadi.csv")


# ==============================
# govt_med UPDATED LOGIC
# ==============================

def detect_category(name):
    name = str(name).lower()

    # --------------------------
    # 1. TABLET FAMILY (GROUPED)
    # --------------------------
    if any(x in name for x in [
        "tablet", "tablets", "tab",
        "film coated tablet",
        "dispersible tablet",
        "chewable tablet"
    ]):
        return "Tablet"

    # --------------------------
    # 2. CAPSULE FAMILY (GROUPED)
    # --------------------------
    elif any(x in name for x in [
        "capsule", "capsules", " cap ", " cap-", "cap "
    ]):
        return "Capsule"

    # --------------------------
    # 3. LIQUID (SEPARATE)
    # --------------------------
    elif "dry syrup" in name:
        return "Dry Syrup"

    elif "oral suspension" in name:
        return "Oral Suspension"

    elif "suspension" in name:
        return "Suspension"

    elif "syrup" in name:
        return "Syrup"

    # --------------------------
    # 4. INJECTION (GROUPED)
    # --------------------------
    elif any(x in name for x in [
        "injection", "infusion"
    ]):
        return "Injection"

    # --------------------------
    # 5. TOPICAL (SEPARATE)
    # --------------------------
    elif "cream" in name:
        return "Cream"

    elif "ointment" in name:
        return "Ointment"

    elif "gel" in name:
        return "Gel"

    # --------------------------
    # 6. DROPS (SEPARATE)
    # --------------------------
    elif "eye drop" in name:
        return "Eye Drops"

    elif "ear drop" in name:
        return "Ear Drops"

    elif "nasal drop" in name:
        return "Nasal Drops"

    elif "drop" in name:
        return "Drops"

    # --------------------------
    # 7. SPECIALIZED (SEPARATE)
    # --------------------------
    elif "inhaler" in name:
        return "Inhaler"

    elif "gargle" in name:
        return "Gargle"

    elif "mouthwash" in name:
        return "Mouthwash"

    # --------------------------
    # 8. TRASH / UNKNOWN
    # --------------------------
    elif any(x in name for x in [
        "powder", "granules", "sachet"
    ]):
        return "Unknown"

    else:
        return "Unknown"


# Apply
jan_df['Header_Category'] = jan_df['Generic Name'].apply(detect_category)


# ==============================
# CHECK OUTPUT
# ==============================

print("\nCategory Distribution:\n")
print(jan_df['Header_Category'].value_counts())

jan_df[['Generic Name', 'Header_Category']].head(20)


Category Distribution:

Header_Category
Tablet             1223
Unknown             507
Injection           211
Capsule             206
Cream                61
Syrup                42
Eye Drops            36
Oral Suspension      33
Gel                  32
Suspension           26
Ointment             21
Drops                12
Inhaler               9
Ear Drops             8
Dry Syrup             5
Nasal Drops           3
Mouthwash             2
Gargle                2
Name: count, dtype: int64


,Generic Name,Header_Category
0,Aceclofenac 100mg and Paracetamol 325mg Tablets,Tablet
1,Aceclofenac Tablets IP 100 mg,Tablet
2,Pregabalin Capsules IP 75 mg,Capsule
3,Aspirin Gastro-resistant Tablets IP 150 mg,Tablet
4,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...",Tablet
5,Diclofenac Gel IP 1.16%w/w (Diclofenac Diethyl...,Gel
6,Serratiopeptidase 10mg and Diclofenac Sodium 5...,Tablet
7,Diclofenac Sodium Prolonged Release Tablets IP...,Tablet
8,Diclofenac Sodium Injection IP 25mg per ml,Injection
9,Diclofenac Gastro-Resistant Tablets IP 50 mg,Tablet


In [26]:
unknown_df = jan_df[jan_df['Header_Category'] == "Unknown"]

print("Total Unknown:", len(unknown_df))

unknown_df[['Generic Name', 'Unit Size']].head(50)

Total Unknown: 507


,Generic Name,Unit Size
105,Calamine Lotion IP,100 ml
109,Ketoconazole Lotion 2% w/v,100 ml Bottle
112,Povidone-Iodine Solution IP 10 % w/v,500 ml
113,Povidone-Iodine Solution IP 5 % w/v,100 ml
114,Povidone Iodine Solution 7.5% w/v,500 ml
157,Lactulose Solution IP 10g per 15ml,100 ml
185,Budesonide Inhalation IP 100mcg,200 MDI
186,Budesonide Inhalation IP 200mcg,200 md
198,Salbutamol Inhalation IP 100mcg,200 md
242,Oral Rehydration Salts IP 20.5g Sachet (WHO Fo...,1's


In [27]:
unknown_df[['Generic Name', 'Unit Size']].sample(50)

,Generic Name,Unit Size
2305,Janaushadhi Bachpan (Baby Diaper (New Born)),5's
2361,Vein Compression Stocking Thigh length (XXL),1's
2094,Cefepime 250 Mg Inection,1.0 Injection in 1 vial
2125,Absorbent Cotton Wool IP 75 g,1's
2340,Chlorhexidine Gauze Dressing B.P 10cmx10cm (St...,1 pc in a pack
2253,Polyamide (3/8 Conventional Cutting Needle 16m...,1's
2127,Absorbent Cotton Wool IP 500 g,1's
2251,"Silk Suture (3/8 Cir Rb needle 16mm, Length 76...",1's
1607,Chyawanprash Special 500g Jar,500g Jar
2432,Elastic Net Fixator Bandage,2 cm X 3 m in Monopack


In [18]:
jan_df[['Salt 1 Name', 'Salt 2 Name', 'Salt 3 Name', 'Salt 1 Strength']] = \
    jan_df['Generic Name'].apply(parse_jan_generic)

In [28]:
unknown_df['Generic Name'].str.split().str[0].value_counts().head(20)

Generic Name
Janaushadhi       56
Surgical          18
Polypropylene     18
Endotracheal      17
Sterile           13
Vein              10
Suction            9
Levosalbutamol     8
Cervical           7
Ketoconazole       6
Beclomethasone     6
Cotton             6
Chromic            6
Polyamide          6
Elastic            6
I.V                5
Ryles              5
Silk               5
Polybutylate       5
Lumbo-Sacral       5
Name: count, dtype: int64

In [29]:
janaushadhi_unknown = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("janaushadhi", case=False, na=False))
]

print("Total Janaushadhi Unknown:", len(janaushadhi_unknown))

janaushadhi_unknown[['Generic Name', 'Unit Size']].head(50)

Total Janaushadhi Unknown: 56


,Generic Name,Unit Size
802,Janaushadhi Protein Powder 250g,1's Tin 250g
803,Janaushadhi Nirmal (Nicotine Polacrilex Chewin...,1 x 9's (mono carton pack)
804,Janaushadhi Urja (Glucose Powder (Orange Flavo...,300 g Box
805,Janaushadhi Urja (Glucose Powder (Orange Flavo...,100 gm Tetra pack
806,Janaushadhi Urja (Glucose Powder (Orange Flavo...,500g Screw-Cap jar with scoop
809,Janaushadhi Janani 250g,1's Tin 250g
821,"Janaushadhi Poshan (Malt based food, 500gm)",1's Screw Cap Plastic Jar
826,Janaushadhi Protein Bar (35 gm),1's Bar
827,Janaushadhi Immunity Bar (10 gm),1's Bar
828,Janaushadhi Protein Powder (Vanilla) 250 gm,1's Tin 250g


In [30]:
# ==============================
# STEP 1: CREATE TRASH DATA
# ==============================

trash_janaushadhi = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("janaushadhi", case=False, na=False))
].copy()

print("Moved to Trash:", len(trash_janaushadhi))


# ==============================
# STEP 2: SAVE TRASH FILE
# ==============================

trash_janaushadhi.to_excel("Trash_Janaushadhi.xlsx", index=False)

print("✅ Trash file saved: Trash_Janaushadhi.xlsx")


# ==============================
# STEP 3: REMOVE FROM MAIN DATA
# ==============================

jan_df = jan_df.drop(trash_janaushadhi.index)

print("Remaining rows in main dataset:", len(jan_df))

Moved to Trash: 56
✅ Trash file saved: Trash_Janaushadhi.xlsx
Remaining rows in main dataset: 2383


In [31]:
surgical_unknown = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("surgical", case=False, na=False))
]

print("Total Surgical Unknown:", len(surgical_unknown))

surgical_unknown[['Generic Name', 'Unit Size']].head(50)

Total Surgical Unknown: 21


,Generic Name,Unit Size
1859,Surgical Spirit IP (Methyl Salicylate 0.5% v/v...,100ml
2151,"Surgical Blade, No. 22, Sterilized",1's
2152,"Surgical Blade, No. 15, Sterilized",1's
2188,"Surgical Rubber Gloves-Disposable, Sterile, 6....",PAIR
2189,"Surgical Rubber Gloves-Disposable, Sterile, 7 ...",PAIR
2190,"Surgical Rubber Gloves-Disposable, Sterile, 7....",PAIR
2191,"Surgical Rubber Gloves-Disposable, Sterile, 8 ...",PAIR
2194,"Surgical Cap, Disposable(for Surgeons/Nurses)",1's
2240,"Surgical Suture (Absorbable, Synthetic) 1/2 Ci...",1's
2241,"Surgical Suture (Absorbable, Synthetic) 1/2 Ci...",1's


In [33]:
# ==============================
# STEP 1: CREATE SURGICAL TRASH
# ==============================

trash_surgical = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("surgical", case=False, na=False))
].copy()

print("Moved to Trash (Surgical):", len(trash_surgical))


# ==============================
# STEP 2: SAVE FILE
# ==============================

trash_surgical.to_excel("Trash_Surgical.xlsx", index=False)

print("✅ Saved: Trash_Surgical.xlsx")


# ==============================
# STEP 3: REMOVE FROM MAIN DATA
# ==============================

jan_df = jan_df.drop(trash_surgical.index)

print("Remaining rows in main dataset:", len(jan_df))

Moved to Trash (Surgical): 21
✅ Saved: Trash_Surgical.xlsx
Remaining rows in main dataset: 2362


In [34]:
poly_unknown = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("polypropylene", case=False, na=False))
]

print("Total Polypropylene Unknown:", len(poly_unknown))

poly_unknown[['Generic Name', 'Unit Size']].head(50)

Total Polypropylene Unknown: 18


,Generic Name,Unit Size
2258,"Polypropylene Suture (1/2 Cir Rb, 13mm Needle,...",1's
2259,"Polypropylene Suture (3/8 Cir Rb, 8mm Needle, ...",1's
2260,"Polypropylene Suture (3/8 Cir Rb, 16mm Needle,...",1's
2261,"Polypropylene Suture (1/2 Cir Rb, 30mm Needle,...",1's
2262,"Polypropylene Suture (1/2 Cir Rb, 45mm Needle,...",1's
2263,"Polypropylene Suture (1/2 Cir Rb, 17mm Needle,...",1's
2264,"Polypropylene Suture (1/2 Tapercut, 17mm Doubl...",1's
2265,"Polypropylene Suture (1/2 Cir Rb, 25mm Needle,...",1's
2266,"Polypropylene Suture (1/2 Cir Rb, 30mm Needle,...",1's
2267,"Polypropylene Suture (1/2 Tapercut, 17mm Doubl...",1's


In [35]:
# ==============================
# STEP 1: CREATE POLYPROPYLENE TRASH
# ==============================

trash_polypropylene = jan_df[
    (jan_df['Header_Category'] == "Unknown") &
    (jan_df['Generic Name'].str.contains("polypropylene", case=False, na=False))
].copy()

print("Moved to Trash (Polypropylene):", len(trash_polypropylene))


# ==============================
# STEP 2: SAVE FILE
# ==============================

trash_polypropylene.to_excel("Trash_Polypropylene.xlsx", index=False)

print("✅ Saved: Trash_Polypropylene.xlsx")


# ==============================
# STEP 3: REMOVE FROM MAIN DATA
# ==============================

jan_df = jan_df.drop(trash_polypropylene.index)

print("Remaining rows in main dataset:", len(jan_df))

Moved to Trash (Polypropylene): 18
✅ Saved: Trash_Polypropylene.xlsx
Remaining rows in main dataset: 2344


In [36]:
remaining_unknown = jan_df[jan_df['Header_Category'] == "Unknown"]

print("Remaining Unknown:", len(remaining_unknown))

remaining_unknown[['Generic Name']].sample(30)

Remaining Unknown: 412


,Generic Name
2308,Knee Brace Belt (Large)
1085,Salmeterol 50mcg and Fluticasone Propionate 10...
2213,"Infant Feeding Tube, Sterile (Size-5 Fg, lengt..."
2318,Nebulizer Mask with Tubing
540,Clotrimazole 1%w/v and Beclometasone Dipropion...
833,Hand Sanitizer 500 ml (Each pack contains: Eth...
2331,Baby Feeding Bottle 250 ml
112,Povidone-Iodine Solution IP 10 % w/v
1264,Potassium Nitrate Toothpaste 5% w/w
832,Hand Sanitizer 100 ml (Each pack contains: Eth...


In [37]:
# ==============================
# STEP 1: EXTRACT ALL REMAINING UNKNOWN
# ==============================

trash_remaining_unknown = jan_df[
    jan_df['Header_Category'] == "Unknown"
].copy()

print("Moved to Trash (Remaining Unknown):", len(trash_remaining_unknown))


# ==============================
# STEP 2: SAVE FILE
# ==============================

trash_remaining_unknown.to_excel("Trash_Remaining_Unknown.xlsx", index=False)

print("✅ Saved: Trash_Remaining_Unknown.xlsx")


# ==============================
# STEP 3: REMOVE FROM MAIN DATA
# ==============================

jan_df = jan_df.drop(trash_remaining_unknown.index)

print("Remaining rows in main dataset:", len(jan_df))

Moved to Trash (Remaining Unknown): 412
✅ Saved: Trash_Remaining_Unknown.xlsx
Remaining rows in main dataset: 1932


In [39]:
import re

# ==============================
# FUNCTION: CLEAN + EXTRACT SALT + STRENGTH
# ==============================

def extract_salt_strength(text):
    text = str(text).lower()

    # --------------------------
    # NORMALIZE MULTI-SALT SEPARATOR
    # --------------------------
    text = text.replace(" and ", " + ")

    # --------------------------
    # REMOVE DOSAGE FORMS COMPLETELY
    # --------------------------
    forms = [
        r'\btablets?\b', r'\btab\b',
        r'\bcapsules?\b', r'\bcap\b',
        r'\bsyrup\b', r'\bsuspension\b', r'\boral suspension\b', r'\bdry syrup\b',
        r'\binjection\b', r'\binfusion\b',
        r'\bcream\b', r'\bgel\b', r'\bointment\b',
        r'\bdrops?\b', r'\beye drops\b', r'\bear drops\b', r'\bnasal drops\b',
        r'\binhaler\b', r'\bgargle\b', r'\bmouthwash\b'
    ]

    for f in forms:
        text = re.sub(f, ' ', text)

    # --------------------------
    # REMOVE DESCRIPTORS (CRITICAL FIX)
    # --------------------------
    descriptors = [
        "gastro-resistant",
        "prolonged release",
        "extended release",
        "sustained release",
        "film coated",
        "dispersible",
        "chewable"
    ]

    for d in descriptors:
        text = text.replace(d, " ")

    # --------------------------
    # REMOVE PHARMA NOISE
    # --------------------------
    noise_words = ["ip", "bp", "usp"]
    for w in noise_words:
        text = re.sub(rf'\b{w}\b', ' ', text)

    # normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # --------------------------
    # ENHANCED EXTRACTION
    # --------------------------
    pattern = r'''
        ([a-zA-Z\s]+?)                 # salt name
        \s*
        (
            \d+\.?\d*\s*              # number
            (mg|ml|g|gm|mcg|iu|%)     # unit
            (
                /\s*\d+\s*(ml|g)      # mg/5 ml, mg/g
            )?
            (
                \s*(w/w|w/v|v/v)      # % w/w etc
            )?
        )
    '''

    matches = re.findall(pattern, text, re.VERBOSE)

    if matches:
        result = []
        for match in matches:
            salt = match[0].strip()
            strength = match[1].strip()

            # remove trailing 's' safely
            salt = re.sub(r'\b([a-z]+)s\b', r'\1', salt)

            result.append(f"{salt} {strength}")

        return " + ".join(result)

    return "NA"


# ==============================
# APPLY TO DATAFRAME
# ==============================

jan_df['salt_and_their_strength'] = jan_df['Generic Name'].apply(extract_salt_strength)


# ==============================
# CHECK OUTPUT
# ==============================

jan_df[['Generic Name', 'salt_and_their_strength']].head(20)

,Generic Name,salt_and_their_strength
0,Aceclofenac 100mg and Paracetamol 325mg Tablets,aceclofenac 100mg + paracetamol 325mg
1,Aceclofenac Tablets IP 100 mg,aceclofenac 100 mg
2,Pregabalin Capsules IP 75 mg,pregabalin 75 mg
3,Aspirin Gastro-resistant Tablets IP 150 mg,aspirin 150 mg
4,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...",chlorzoxazone 500mg + diclofenac 50mg + parace...
5,Diclofenac Gel IP 1.16%w/w (Diclofenac Diethyl...,diclofenac 1.16%w/w
6,Serratiopeptidase 10mg and Diclofenac Sodium 5...,serratiopeptidase 10mg + diclofenac sodium 50mg
7,Diclofenac Sodium Prolonged Release Tablets IP...,diclofenac sodium 100 mg
8,Diclofenac Sodium Injection IP 25mg per ml,diclofenac sodium 25mg
9,Diclofenac Gastro-Resistant Tablets IP 50 mg,diclofenac 50 mg


In [41]:
import re
import pandas as pd

# ==============================
# STEP 1: EXTRACT SALT + STRENGTH
# ==============================

def extract_salt_strength(text):
    text = str(text).lower()

    # normalize "and" → "+"
    text = text.replace(" and ", " + ")

    # remove dosage forms
    forms = [
        r'\btablets?\b', r'\btab\b',
        r'\bcapsules?\b', r'\bcap\b',
        r'\bsyrup\b', r'\bsuspension\b', r'\boral suspension\b', r'\bdry syrup\b',
        r'\binjection\b', r'\binfusion\b',
        r'\bcream\b', r'\bgel\b', r'\bointment\b',
        r'\bdrops?\b', r'\beye drops\b', r'\bear drops\b', r'\bnasal drops\b',
        r'\binhaler\b', r'\bgargle\b', r'\bmouthwash\b'
    ]

    for f in forms:
        text = re.sub(f, ' ', text)

    # remove descriptors
    descriptors = [
        "gastro-resistant", "prolonged release",
        "extended release", "sustained release",
        "film coated", "dispersible", "chewable"
    ]

    for d in descriptors:
        text = text.replace(d, " ")

    # remove pharma noise
    for w in ["ip", "bp", "usp"]:
        text = re.sub(rf'\b{w}\b', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    # extract salt + strength
    pattern = r'''
        ([a-zA-Z\s]+?)
        \s*
        (
            \d+\.?\d*\s*
            (mg|ml|g|gm|mcg|iu|%)
            (
                /\s*\d+\s*(ml|g)
            )?
            (
                \s*(w/w|w/v|v/v)
            )?
        )
    '''

    matches = re.findall(pattern, text, re.VERBOSE)

    if matches:
        result = []
        for match in matches:
            salt = match[0].strip()
            strength = match[1].strip()

            # remove trailing 's'
            salt = re.sub(r'\b([a-z]+)s\b', r'\1', salt)

            result.append(f"{salt} {strength}")

        return " + ".join(result)

    return "NA"


# ==============================
# STEP 2: STANDARDIZE + CONVERT
# ==============================

def standardize_strength(strength):
    if strength == "NA":
        return "NA"

    strength = strength.lower().strip()

    # spacing
    strength = re.sub(r'(\d)(mg|ml|g|gm|mcg|iu|%)', r'\1 \2', strength)
    strength = re.sub(r'/(\s*)(\d+)(ml|g)', r'/\2 \3', strength)
    strength = re.sub(r'%\s*(w/w|w/v|v/v)', r'% \1', strength)

    # --------------------------
    # CONVERSION LOGIC
    # --------------------------

    # g → mg
    strength = re.sub(r'(\d+\.?\d*)\s*g\b', lambda m: f"{float(m.group(1))*1000} mg", strength)

    # mcg → mg
    strength = re.sub(r'(\d+\.?\d*)\s*mcg\b', lambda m: f"{float(m.group(1))/1000} mg", strength)

    # --------------------------
    # HANDLE mg/5 ml → mg/ml
    # --------------------------
    match = re.match(r'(\d+\.?\d*) mg/(\d+\.?\d*) ml', strength)
    if match:
        mg = float(match.group(1))
        ml = float(match.group(2))
        value = mg / ml
        strength = f"{value} mg/ml"

    # clean spacing
    strength = re.sub(r'\s+', ' ', strength)

    return strength.strip()


# ==============================
# STEP 3: SPLIT SALTS
# ==============================

def split_salts(row):
    text = row['salt_and_their_strength']

    if text == "NA":
        return pd.Series(["NA", "NA", "NA", "NA", "NA", "NA"])

    parts = text.split("+")

    salts = []
    strengths = []

    for part in parts:
        part = part.strip()

        match = re.match(r'(.+?)\s*(\d+.*)', part)

        if match:
            salt = match.group(1).strip()
            strength = match.group(2).strip()
            strength = standardize_strength(strength)
        else:
            salt = part
            strength = "NA"

        salts.append(salt)
        strengths.append(strength)

    while len(salts) < 3:
        salts.append("NA")
        strengths.append("NA")

    return pd.Series([
        salts[0], strengths[0],
        salts[1], strengths[1],
        salts[2], strengths[2]
    ])


# ==============================
# APPLY PIPELINE
# ==============================

jan_df['salt_and_their_strength'] = jan_df['Generic Name'].apply(extract_salt_strength)

jan_df[[
    'Salt 1 Name', 'Salt 1 Strength',
    'Salt 2 Name', 'Salt 2 Strength',
    'Salt 3 Name', 'Salt 3 Strength'
]] = jan_df.apply(split_salts, axis=1)


# ==============================
# CHECK
# ==============================

jan_df[[
    'Generic Name',
    'salt_and_their_strength',
    'Salt 1 Name', 'Salt 1 Strength'
]].head(20)

,Generic Name,salt_and_their_strength,Salt 1 Name,Salt 1 Strength
0,Aceclofenac 100mg and Paracetamol 325mg Tablets,aceclofenac 100mg + paracetamol 325mg,aceclofenac,100 mg
1,Aceclofenac Tablets IP 100 mg,aceclofenac 100 mg,aceclofenac,100 mg
2,Pregabalin Capsules IP 75 mg,pregabalin 75 mg,pregabalin,75 mg
3,Aspirin Gastro-resistant Tablets IP 150 mg,aspirin 150 mg,aspirin,150 mg
4,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...",chlorzoxazone 500mg + diclofenac 50mg + parace...,chlorzoxazone,500 mg
5,Diclofenac Gel IP 1.16%w/w (Diclofenac Diethyl...,diclofenac 1.16%w/w,diclofenac,1.16 % w/w
6,Serratiopeptidase 10mg and Diclofenac Sodium 5...,serratiopeptidase 10mg + diclofenac sodium 50mg,serratiopeptidase,10 mg
7,Diclofenac Sodium Prolonged Release Tablets IP...,diclofenac sodium 100 mg,diclofenac sodium,100 mg
8,Diclofenac Sodium Injection IP 25mg per ml,diclofenac sodium 25mg,diclofenac sodium,25 mg
9,Diclofenac Gastro-Resistant Tablets IP 50 mg,diclofenac 50 mg,diclofenac,50 mg


In [45]:
jan_df.info()


<class 'pandas.DataFrame'>
Index: 1932 entries, 0 to 2431
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Sr No                    1932 non-null   int64  
 1   Drug Code                1932 non-null   int64  
 2   Generic Name             1932 non-null   str    
 3   Unit Size                1932 non-null   str    
 4   MRP                      1932 non-null   float64
 5   Group Name               1932 non-null   str    
 6   Header_Category          1932 non-null   str    
 7   salt_and_their_strength  1932 non-null   str    
 8   Salt 1 Name              1932 non-null   str    
 9   Salt 1 Strength          1932 non-null   str    
 10  Salt 2 Name              1932 non-null   str    
 11  Salt 2 Strength          1932 non-null   str    
 12  Salt 3 Name              1932 non-null   str    
 13  Salt 3 Strength          1932 non-null   str    
dtypes: float64(1), int64(2), str(11)
memory 

In [46]:
jan_df.insert(1, 'name', jan_df['Generic Name'])

In [47]:
jan_df[['Generic Name', 'name']].head()

,Generic Name,name
0,Aceclofenac 100mg and Paracetamol 325mg Tablets,Aceclofenac 100mg and Paracetamol 325mg Tablets
1,Aceclofenac Tablets IP 100 mg,Aceclofenac Tablets IP 100 mg
2,Pregabalin Capsules IP 75 mg,Pregabalin Capsules IP 75 mg
3,Aspirin Gastro-resistant Tablets IP 150 mg,Aspirin Gastro-resistant Tablets IP 150 mg
4,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...","Chlorzoxazone 500mg, Diclofenac 50mg and Parac..."


In [48]:
jan_df.info()


<class 'pandas.DataFrame'>
Index: 1932 entries, 0 to 2431
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Sr No                    1932 non-null   int64  
 1   name                     1932 non-null   str    
 2   Drug Code                1932 non-null   int64  
 3   Generic Name             1932 non-null   str    
 4   Unit Size                1932 non-null   str    
 5   MRP                      1932 non-null   float64
 6   Group Name               1932 non-null   str    
 7   Header_Category          1932 non-null   str    
 8   salt_and_their_strength  1932 non-null   str    
 9   Salt 1 Name              1932 non-null   str    
 10  Salt 1 Strength          1932 non-null   str    
 11  Salt 2 Name              1932 non-null   str    
 12  Salt 2 Strength          1932 non-null   str    
 13  Salt 3 Name              1932 non-null   str    
 14  Salt 3 Strength          1932 non-null  

In [50]:
import re
import pandas as pd

# ==============================
# FUNCTION: EXTRACT QUANTITY (FINAL)
# ==============================

def extract_quantity(row):
    text = str(row['Unit Size']).lower()
    category = str(row['Header_Category']).lower()

    # --------------------------
    # CLEAN TEXT
    # --------------------------
    text = re.sub(r'\(.*?\)', ' ', text)   # remove brackets
    text = re.sub(r'\s+', ' ', text).strip()

    # ==========================
    # 1. INJECTION LOGIC (PRIORITY)
    # ==========================
    if "injection" in category:

        # Case 1: 5 ampoules / 1 vial
        match = re.search(r'(\d+)\s*(ampoule|ampoules|vial|vials)', text)
        if match:
            return int(match.group(1))

        # Case 2: 2 ml vial / 10 ml ampoule
        if "vial" in text or "ampoule" in text:
            return 1

        # Case 3: ONLY ml present → treat as 1 unit
        if re.search(r'\d+\.?\d*\s*ml', text):
            return 1

    # ==========================
    # 2. WEIGHT (g / gm)
    # ==========================
    match = re.search(r'(\d+\.?\d*)\s*(g|gm)', text)
    if match:
        return float(match.group(1))

    # ==========================
    # 3. VOLUME (ml) — NON-INJECTION
    # ==========================
    match = re.search(r'(\d+\.?\d*)\s*ml', text)
    if match:
        return float(match.group(1))

    # ==========================
    # 4. COUNT-BASED
    # ==========================

    # 1 x 10's
    match = re.search(r'1\s*x\s*(\d+)', text)
    if match:
        return int(match.group(1))

    # Pack of 10
    match = re.search(r'pack of (\d+)', text)
    if match:
        return int(match.group(1))

    # 10's
    match = re.search(r'(\d+)\s*\'?s', text)
    if match:
        return int(match.group(1))

    # fallback number
    match = re.search(r'\b(\d+)\b', text)
    if match:
        return int(match.group(1))

    # ==========================
    # DEFAULT
    # ==========================
    return None


# ==============================
# APPLY TO DATAFRAME
# ==============================

jan_df['quantity'] = jan_df.apply(extract_quantity, axis=1)


# ==============================
# CHECK OUTPUT
# ==============================

jan_df[['Header_Category', 'Unit Size', 'quantity']].head(30)

,Header_Category,Unit Size,quantity
0,Tablet,10's,10.0
1,Tablet,10's,10.0
2,Capsule,10's,10.0
3,Tablet,14's,14.0
4,Tablet,10's,10.0
5,Gel,15 g,15.0
6,Tablet,10's,10.0
7,Tablet,10's,10.0
8,Injection,3 ml,1.0
9,Tablet,10's,10.0


In [51]:
import re
import pandas as pd

# ==============================
# FUNCTION: EXTRACT QUANTITY + UNIT TYPE
# ==============================

def extract_quantity_and_unit(row):
    text = str(row['Unit Size']).lower()
    category = str(row['Header_Category']).lower()

    # clean text
    text = re.sub(r'\(.*?\)', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # ==========================
    # 1. INJECTION LOGIC
    # ==========================
    if "injection" in category:

        # 5 ampoules / 1 vial
        match = re.search(r'(\d+)\s*(ampoule|ampoules|vial|vials)', text)
        if match:
            return pd.Series([int(match.group(1)), "unit"])

        # 2 ml vial / 10 ml ampoule
        if "vial" in text or "ampoule" in text:
            return pd.Series([1, "unit"])

        # ONLY ml → still 1 unit
        if re.search(r'\d+\.?\d*\s*ml', text):
            return pd.Series([1, "unit"])

    # ==========================
    # 2. WEIGHT (g)
    # ==========================
    match = re.search(r'(\d+\.?\d*)\s*(g|gm)', text)
    if match:
        return pd.Series([float(match.group(1)), "g"])

    # ==========================
    # 3. VOLUME (ml)
    # ==========================
    match = re.search(r'(\d+\.?\d*)\s*ml', text)
    if match:
        return pd.Series([float(match.group(1)), "ml"])

    # ==========================
    # 4. COUNT
    # ==========================

    # 1 x 10's
    match = re.search(r'1\s*x\s*(\d+)', text)
    if match:
        return pd.Series([int(match.group(1)), "count"])

    # Pack of 10
    match = re.search(r'pack of (\d+)', text)
    if match:
        return pd.Series([int(match.group(1)), "count"])

    # 10's
    match = re.search(r'(\d+)\s*\'?s', text)
    if match:
        return pd.Series([int(match.group(1)), "count"])

    # fallback number → assume count
    match = re.search(r'\b(\d+)\b', text)
    if match:
        return pd.Series([int(match.group(1)), "count"])

    # ==========================
    # DEFAULT
    # ==========================
    return pd.Series([None, "unknown"])


# ==============================
# APPLY TO DATAFRAME
# ==============================

jan_df[['quantity', 'unit_type']] = jan_df.apply(extract_quantity_and_unit, axis=1)


# ==============================
# CHECK OUTPUT
# ==============================

jan_df[['Header_Category', 'Unit Size', 'quantity', 'unit_type']].head(30)

,Header_Category,Unit Size,quantity,unit_type
0,Tablet,10's,10.0,count
1,Tablet,10's,10.0,count
2,Capsule,10's,10.0,count
3,Tablet,14's,14.0,count
4,Tablet,10's,10.0,count
5,Gel,15 g,15.0,g
6,Tablet,10's,10.0,count
7,Tablet,10's,10.0,count
8,Injection,3 ml,1.0,unit
9,Tablet,10's,10.0,count


In [54]:
# ==============================
# FUNCTION: CALCULATE PRICE PER UNIT
# ==============================

def calculate_price_per_unit(row):
    price = row.get('MRP')   # ← using correct column
    quantity = row.get('quantity')

    if price is None or quantity is None:
        return None

    try:
        price = float(price)

        if quantity == 0:
            return None
        
        return round(price / quantity, 3)
    
    except:
        return None


# ==============================
# APPLY TO DATAFRAME
# ==============================

jan_df['price_per_unit'] = jan_df.apply(calculate_price_per_unit, axis=1)


# ==============================
# CHECK OUTPUT
# ==============================

jan_df[['MRP', 'quantity', 'unit_type', 'price_per_unit']].head(50)

,MRP,quantity,unit_type,price_per_unit
0,9.38,10.0,count,0.938
1,7.50,10.0,count,0.750
2,20.63,10.0,count,2.063
3,4.69,14.0,count,0.335
4,23.44,10.0,count,2.344
5,11.25,15.0,g,0.750
6,14.44,10.0,count,1.444
7,11.35,10.0,count,1.135
8,3.75,1.0,unit,3.750
9,5.16,10.0,count,0.516


In [56]:
jan_df[['Header_Category','Unit Size','MRP','quantity','unit_type','price_per_unit']].sample(20)

,Header_Category,Unit Size,MRP,quantity,unit_type,price_per_unit
1327,Tablet,10's,24.47,10.0,count,2.447
422,Tablet,10's,41.25,10.0,count,4.125
1024,Tablet,10's,28.13,10.0,count,2.813
1551,Cream,30gm Lami Tube,0.00,30.0,g,0.000
1883,Tablet,10's,0.00,10.0,count,0.000
1186,Tablet,10's,21.57,10.0,count,2.157
175,Tablet,10's,12.19,10.0,count,1.219
1277,Tablet,15's in bottle,0.00,15.0,count,0.000
524,Injection,5 ml,1.41,1.0,unit,1.410
38,Tablet,10's,11.25,10.0,count,1.125


In [57]:
jan_df.head()

,Sr No,name,Drug Code,Generic Name,Unit Size,MRP,Group Name,Header_Category,salt_and_their_strength,Salt 1 Name,Salt 1 Strength,Salt 2 Name,Salt 2 Strength,Salt 3 Name,Salt 3 Strength,quantity,unit_type,price_per_unit
0,1,Aceclofenac 100mg and Paracetamol 325mg Tablets,1,Aceclofenac 100mg and Paracetamol 325mg Tablets,10's,9.38,Analgesic/Antipyretic/Anti-Inflammatory,Tablet,aceclofenac 100mg + paracetamol 325mg,aceclofenac,100 mg,paracetamol,325 mg,NA,NA,10.0,count,0.938
1,2,Aceclofenac Tablets IP 100 mg,2,Aceclofenac Tablets IP 100 mg,10's,7.50,Analgesic/Antipyretic/Anti-Inflammatory,Tablet,aceclofenac 100 mg,aceclofenac,100 mg,NA,NA,NA,NA,10.0,count,0.750
2,3,Pregabalin Capsules IP 75 mg,3,Pregabalin Capsules IP 75 mg,10's,20.63,Central Nervous System (CNS),Capsule,pregabalin 75 mg,pregabalin,75 mg,NA,NA,NA,NA,10.0,count,2.063
3,4,Aspirin Gastro-resistant Tablets IP 150 mg,5,Aspirin Gastro-resistant Tablets IP 150 mg,14's,4.69,Analgesic/Antipyretic/Anti-Inflammatory,Tablet,aspirin 150 mg,aspirin,150 mg,NA,NA,NA,NA,14.0,count,0.335
4,5,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...",6,"Chlorzoxazone 500mg, Diclofenac 50mg and Parac...",10's,23.44,Analgesic/Antipyretic/Anti-Inflammatory,Tablet,chlorzoxazone 500mg + diclofenac 50mg + parace...,chlorzoxazone,500 mg,diclofenac,50 mg,paracetamol,325 mg,10.0,count,2.344


In [58]:
import pandas as pd
import re

# ==============================
# STEP 1: DEFINE REQUIRED COLUMNS
# ==============================

required_columns = [
    "Sr No",
    "name",
    "Generic Name",
    "MRP",
    "Header_Category",
    "salt_and_their_strength",
    "Salt 1 Name",
    "Salt 1 Strength",
    "Salt 2 Name",
    "Salt 2 Strength",
    "Salt 3 Name",
    "Salt 3 Strength",
    "quantity",
    "unit_type",
    "price_per_unit"
]

# keep only columns that exist (safe handling)
required_columns = [col for col in required_columns if col in jan_df.columns]


# ==============================
# STEP 2: DEFINE CATEGORY GROUPS
# ==============================

internal_categories = [
    "Tablet",
    "Capsule",
    "Syrup",
    "Suspension",
    "Oral Suspension",
    "Dry Syrup"
]

external_categories = [
    "Cream",
    "Gel",
    "Ointment",
    "Drops"
]

specialized_categories = [
    "Injection",
    "Inhaler",
    "Gargle"
]


# ==============================
# STEP 3: SPLIT DATAFRAMES
# ==============================

jan_internal_df = jan_df[jan_df['Header_Category'].isin(internal_categories)].copy()
jan_external_df = jan_df[jan_df['Header_Category'].isin(external_categories)].copy()
jan_special_df  = jan_df[jan_df['Header_Category'].isin(specialized_categories)].copy()

# keep only required columns
jan_internal_df = jan_internal_df[required_columns]
jan_external_df = jan_external_df[required_columns]
jan_special_df  = jan_special_df[required_columns]


# ==============================
# STEP 4: CLEAN SHEET NAMES
# ==============================

def clean_sheet_name(name):
    name = str(name)
    name = re.sub(r'[\\/*?:\[\]]', '', name)
    return name[:31]


# ==============================
# STEP 5: SAVE FUNCTION
# ==============================

def save_to_excel(df, filename):
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        
        for category in df['Header_Category'].unique():
            sheet_name = clean_sheet_name(category)
            
            temp_df = df[df['Header_Category'] == category]
            
            temp_df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"✅ Saved: {filename}")


# ==============================
# STEP 6: SAVE FILES
# ==============================

save_to_excel(jan_internal_df, "internal_medicines_jan.xlsx")
save_to_excel(jan_external_df, "external_medicines_jan.xlsx")
save_to_excel(jan_special_df,  "specialised_medicines_jan.xlsx")


# ==============================
# STEP 7: VALIDATION
# ==============================

print("\n🔍 VALIDATION CHECK")

print("Internal rows:", len(jan_internal_df))
print("External rows:", len(jan_external_df))
print("Specialized rows:", len(jan_special_df))

print("\nTotal original rows:", len(jan_df))
print("Total split rows:", len(jan_internal_df) + len(jan_external_df) + len(jan_special_df))

✅ Saved: internal_medicines_jan.xlsx
✅ Saved: external_medicines_jan.xlsx
✅ Saved: specialised_medicines_jan.xlsx

🔍 VALIDATION CHECK
Internal rows: 1535
External rows: 126
Specialized rows: 222

Total original rows: 1932
Total split rows: 1883
